# ZIP-level value outlook

Issue #59. The prototype projected each home's value five years ahead from per-ZIP ARIMA forecasts. Before serving anything forward-looking, this notebook (1) checks what the supplied ZIP series is, and (2) backtests simple forecasting methods against a no-change baseline on rolling origins.

Decision rule: an exploratory run of this same backtest came first, so the rule below is not blind; it was chosen as the conservative option and is applied unchanged here. choose the method with the lowest mean absolute percentage error averaged over 1-5 year horizons on **development origins (January 2006-2015)**. Serve a projection only if that method beats the no-change baseline on development origins **and** on the later **holdout origins (January 2016-2020)** at every horizon. Otherwise serve no forecast.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from homelens.data.inventory import _file_identity

raw_root = Path(os.environ.get("HOMELENS_RAW_ROOT", "data/raw"))
path = raw_root / "zipcode_saleprice.csv"
frame = pd.read_csv(path, dtype={"RegionName": "string"})
identity = _file_identity(path)
month_columns = [column for column in frame.columns if column[:2] == "20"]
print("sha256:", identity["sha256"][:16], "bytes:", identity["file_bytes"])
print("identifier columns:", [c for c in frame.columns if c not in month_columns])
print("ZIPs:", sorted(frame["RegionName"].tolist()))
print("months:", month_columns[0], "to", month_columns[-1], f"({len(month_columns)})")

sha256: 6a2eb91c2498ddf6 bytes: 46832
identifier columns: ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName']
ZIPs: ['27503', '27701', '27703', '27704', '27705', '27707', '27712', '27713']
months: 2001-02-01 to 2025-04-01 (291)


## What the series is

The identifier columns (`RegionID`, `SizeRank`, `RegionName`, `RegionType`, `StateName`, `Metro`, `CountyName`) match the layout of Zillow Research ZIP-level downloads, and the values are smooth monthly dollar levels rather than noisy sale medians, which fits the Zillow Home Value Index (ZHVI). Two details differ from current Zillow downloads, which use month-end dates and start in January 2000, so the file was likely re-exported. Treat it as a **ZHVI-style typical home value index of unconfirmed variant**, not observed median sale prices.

In [2]:
series = frame.set_index("RegionName")[month_columns].T
series.index = pd.to_datetime(series.index)
monthly_change = np.log(series).diff()
print("missing values by ZIP:", series.isna().sum()[series.isna().sum() > 0].to_dict())
print(
    "largest absolute one-month log change:",
    round(float(monthly_change.abs().max().max()), 4),
)
print(
    "share of months with a sign change in growth:",
    round(float((np.sign(monthly_change).diff().abs() > 0).mean().mean()), 3),
)
series = series.interpolate(limit=1, limit_area="inside")
log_value = np.log(series)

missing values by ZIP: {'27713': 1}
largest absolute one-month log change: 0.031
share of months with a sign change in growth: 0.075


The single missing month (27713) is filled by linear interpolation for the backtest only. Monthly changes are small and persistent, consistent with a smoothed index.

## Backtest

Each origin uses only months before it. Forecasts are made in log space for horizons of 12-60 months. Methods: no change (`naive`); linear drift over the full history, 10 years, 5 years, or the last 12 months; a damped 12-month trend (monthly damping 0.98); and ARIMA(2,0,0) with a constant on monthly log changes, the closest simple analogue to the prototype.

In [3]:
import warnings

from statsmodels.tsa.arima.model import ARIMA

HORIZONS = [12, 24, 36, 48, 60]
METHODS = [
    "naive",
    "drift_full",
    "drift_10y",
    "drift_5y",
    "drift_1y",
    "damped_1y",
    "arima",
]


def drift(history, months):
    window = history.iloc[-(months + 1) :]
    return (window.iloc[-1] - window.iloc[0]) / months


def forecast(history, horizon, method):
    last = history.iloc[-1]
    if method == "naive":
        return last
    if method == "drift_full":
        return last + drift(history, len(history) - 1) * horizon
    if method in ("drift_10y", "drift_5y", "drift_1y"):
        months = {"drift_10y": 120, "drift_5y": 60, "drift_1y": 12}[method]
        return last + drift(history, months) * horizon
    if method == "damped_1y":
        return last + drift(history, 12) * sum(
            0.98**step for step in range(1, horizon + 1)
        )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = ARIMA(
            history.diff().dropna().to_numpy(), order=(2, 0, 0), trend="c"
        ).fit()
    return last + float(fitted.forecast(horizon).sum())


rows = []
for origin in pd.date_range("2006-01-01", "2020-01-01", freq="12MS"):
    for zip_code in log_value.columns:
        history = log_value.loc[log_value.index < origin, zip_code]
        for horizon in HORIZONS:
            target = origin + pd.DateOffset(months=horizon - 1)
            if target not in log_value.index:
                continue
            actual = log_value.loc[target, zip_code]
            for method in METHODS:
                error = forecast(history, horizon, method) - actual
                rows.append((origin.year, zip_code, horizon, method, error))
backtest = pd.DataFrame(
    rows, columns=["origin", "zip", "horizon", "method", "log_error"]
)
backtest["abs_pct_error"] = (np.exp(backtest["log_error"]) - 1).abs() * 100
backtest["period"] = np.where(backtest["origin"] <= 2015, "development", "holdout")
print(backtest.groupby("period")["origin"].agg(["min", "max", "count"]))

              min   max  count
period                        
development  2006  2015   2800
holdout      2016  2020   1400


In [4]:
def table(period):
    return (
        backtest[backtest["period"] == period]
        .groupby(["horizon", "method"])["abs_pct_error"]
        .mean()
        .unstack()[METHODS]
        .round(1)
    )


development = table("development")
holdout = table("holdout")
print("Development origins 2006-2015: mean absolute % error")
print(development.to_string())
print()
print("Holdout origins 2016-2020: mean absolute % error")
print(holdout.to_string())

Development origins 2006-2015: mean absolute % error
method   naive  drift_full  drift_10y  drift_5y  drift_1y  damped_1y  arima
horizon                                                                    
12         4.4         4.6        4.7       4.8       3.9        3.7    3.3
24         7.9         9.1        9.0      10.0       8.0        7.4    8.2
36        11.1        14.1       13.2      15.4      11.6       10.8   13.1
48        14.2        18.6       17.4      20.5      17.0       15.1   17.6
60        17.3        22.2       20.5      25.3      22.4       19.7   21.6

Holdout origins 2016-2020: mean absolute % error
method   naive  drift_full  drift_10y  drift_5y  drift_1y  damped_1y  arima
horizon                                                                    
12         8.0         6.0        6.1       3.1       2.3        2.4    3.1
24        16.8        13.0       13.0       7.3       5.2        6.4    9.5
36        25.4        20.3       20.3      12.1       8.7    

In [5]:
selected = development.mean().idxmin()
print("development mean over horizons:", development.mean().round(2).to_dict())
print("selected by the fixed rule:", selected)
beats_naive_dev = bool((development[selected] < development["naive"]).all())
beats_naive_hold = bool((holdout[selected] < holdout["naive"]).all())
print("beats no-change at every horizon, development:", beats_naive_dev)
print("beats no-change at every horizon, holdout:", beats_naive_hold)
serve_forecast = selected != "naive" and beats_naive_dev and beats_naive_hold
print("serve a forward projection:", serve_forecast)

development mean over horizons: {'naive': 10.98, 'drift_full': 13.72, 'drift_10y': 12.96, 'drift_5y': 15.2, 'drift_1y': 12.58, 'damped_1y': 11.34, 'arima': 12.76}
selected by the fixed rule: naive
beats no-change at every horizon, development: False
beats no-change at every horizon, holdout: False
serve a forward projection: False


In [6]:
signed = (
    backtest.groupby(["period", "horizon", "method"])["log_error"]
    .mean()
    .unstack()[METHODS]
)
print("Mean signed log error (negative = forecast too low):")
print(signed.round(3).to_string())

Mean signed log error (negative = forecast too low):
method               naive  drift_full  drift_10y  drift_5y  drift_1y  damped_1y  arima
period      horizon                                                                    
development 12      -0.008       0.007      0.001    -0.005    -0.005     -0.005 -0.000
            24      -0.023       0.008     -0.004    -0.017    -0.016     -0.017 -0.003
            36      -0.042       0.004     -0.015    -0.033    -0.032     -0.035 -0.010
            48      -0.073      -0.012     -0.036    -0.062    -0.060     -0.065 -0.027
            60      -0.113      -0.037     -0.067    -0.098    -0.096     -0.103 -0.053
holdout     12      -0.083      -0.061     -0.061    -0.023    -0.005     -0.014 -0.029
            24      -0.186      -0.141     -0.140    -0.066    -0.029     -0.062 -0.102
            36      -0.297      -0.230     -0.229    -0.118    -0.061     -0.131 -0.188
            48      -0.396      -0.307     -0.305    -0.157    -0.0

## Result

At five years, average misses range from 12% to 38% depending on method and period, and which method looks best depends on the period. On development origins, which include the 2008 downturn, no change is the most accurate method on average across 1-5 years, so the fixed rule selects no change: no method earns a forward projection. On holdout origins, which end in the 2020-2022 run-up, trend methods look better because prices rose faster than any baseline assumed, which is the regime dependence this rule is meant to guard against. ARIMA, as in the prototype, is best only at one year on development origins and underperforms trend methods on the holdout.

Decision: **serve no forward projection.** The application may show the ZIP index's recorded history and the recorded sale price moved along that history to the latest index month, labeled as an index adjustment rather than a valuation. Revisit with a longer, confirmed Zillow series (current downloads extend past April 2025) and a pre-registered method before serving any forecast.